# DCF Valuation

Estimates the **intrinsic value per share** of each investable company with a discounted cash flow (DCF) model and compares it with the market price.

It uses the table built and saved by `01_fcf_history.ipynb` (free cash flow, net debt and shares outstanding), so run that notebook first.

| Module | Contents |
|---|---|
| `tickers.py` | `prepare_tickers`, `build_tickers` |
| `prices.py` | `fetch_market_data`, `pick_main_ticker` |
| `valuation.py` | `intrinsic_value`, `build_valuation`, `sensitivity_table` |

> The result of a DCF depends heavily on its assumptions. The numbers below are a starting point for analysis, not investment advice.

## 1. Setup

In [3]:
import pandas as pd

from cvm_valuation.config import (
    DATA_DIR, DFP_URL, FCA_URL,
    DISCOUNT_RATE, FCF_BASE_YEARS, GROWTH_RATE, PROJECTION_YEARS, TERMINAL_GROWTH,
)
from cvm_valuation.sources import find_zips, complete_years
from cvm_valuation.tickers import build_tickers
from cvm_valuation.prices import fetch_market_data, pick_main_ticker
from cvm_valuation.valuation import build_valuation, intrinsic_value, sensitivity_table

pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load the history

One row per company and year, with `FCF`, `NET_DEBT` and `SHARES` (see `01_fcf_history.ipynb`).

In [4]:
history_full = pd.read_csv(DATA_DIR / "history_full.csv")
latest_year = history_full["YEAR"].max()
investable = history_full[(history_full["YEAR"] == latest_year) & history_full["IS_INVESTABLE"]]
print(f"Latest year: {latest_year}, investable companies: {len(investable)}")

Latest year: 2025, investable companies: 221


## 3. Tickers (`tickers.py`)

The DFP has no ticker. The tickers come from the **FCA** (Formulário Cadastral), in its securities file (`valor_mobiliario`). The FCA describes the company today, so the latest zip is used even if the year is not over.

`prepare_tickers` keeps common and preferred shares listed on the exchange (`Mercado = "Bolsa"`), still traded (no `Data_Fim_Negociacao`) and with a valid code (four characters plus one or two digits). Cases found in the 2026 file:

| Problem | Example | Fix |
|---|---|---|
| Debt and other securities | Petrobras debentures and commercial notes | Keep only common and preferred shares |
| Tickers no longer traded | BRFS3, ended in September 2025 (BRF merged with Marfrig) | Drop rows with an end-of-trading date |
| Invalid codes | `000000`, `RBNS`, `NÃO HÁ` | Keep codes matching `TICKER_PATTERN` |
| Units (bundles of shares) | ENGI11 = 1 common + 4 preferred shares | Left out: every company with units also lists its common and preferred shares |

The FCA identifies companies only by CNPJ, so `build_tickers` takes the CVM code from the balance sheet of the latest DFP, as in `shares.py`.

In [5]:
fca_links = find_zips(FCA_URL)
dfp_links = complete_years(find_zips(DFP_URL))
tickers = build_tickers(fca_links, dfp_links)
tickers = tickers[tickers["CD_CVM"].isin(investable["CD_CVM"])]
print(f"{len(tickers)} tickers of {tickers['CD_CVM'].nunique()} investable companies")
tickers.head()

256 tickers of 202 investable companies


,CD_CVM,TICKER,SHARE_TYPE
1,2437,AXIA3,Ações Ordinárias
2,2437,AXIA7,Ações Preferenciais
6,26174,AESO3,Ações Ordinárias
7,26069,ARML3,Ações Ordinárias
10,15253,ENGI3,Ações Ordinárias


## 4. Prices (`prices.py`)

`fetch_market_data` downloads three months of daily prices from Yahoo Finance (B3 tickers end in `.SA`):

- `PRICE` is the last close, **not adjusted** for dividends, so it is comparable with a value per share computed from today's share count.
- `AVG_TRADED_VALUE` is the average daily traded value (close x volume), in reais: a measure of liquidity.

A company may list more than one class (PETR3 and PETR4). The model gives every share of the company the same value, so `pick_main_ticker` keeps the most traded class, which is not always the common share.

In [6]:
market_all = fetch_market_data(tickers["TICKER"].tolist())
market = pick_main_ticker(tickers, market_all)
print(f"Tickers without price: {market_all['PRICE'].isna().sum()}")
print(f"Companies with a price: {len(market)}")

$QUSW3.SA: No data found, symbol may be delisted
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NEMO5.SA"}}}
$NEMO5.SA: No data found, symbol may be delisted
$AESO3.SA: No data found, symbol may be delisted
$ODER3.SA: No data found, symbol may be delisted
$TRXR3.SA: No data found, symbol may be delisted
$CRDE3.SA: No data found, symbol may be delisted
$PTCA3.SA: No data found, symbol may be delisted
$RSUL3.SA: No data found, symbol may be delisted
$CTCA3.SA: No data found, symbol may be delisted
$BBML3.SA: No data found, symbol may be delisted
$MGEL3.SA: No data found, symbol may be delisted
$NEMO6.SA: No data found, symbol may be delisted
$NEMO3.SA: No data found, symbol may be delisted
$INNC3.SA: No data found, symbol may be delisted
$APTI4.SA: No data found, symbol may be delisted
$APTI3.SA: No data found, symbol may be delisted

16 Failed downloads:
['QUSW3.SA', 'NEMO5.SA', 'AESO3.SA', 'ODER3.SA', 'TRXR3.SA', 'C

Tickers without price: 16
Companies with a price: 193


## 5. The DCF model (`valuation.py`)

A two-stage DCF of the free cash flow to the firm:

1. **Base FCF**: the average FCF of the last `FCF_BASE_YEARS` years, to smooth one-off years. Companies without FCF in all of them, or with a non-positive average, are not valued.
2. **Explicit stage**: the base FCF grows at `GROWTH_RATE` for `PROJECTION_YEARS` years.
3. **Terminal value**: after that, FCF grows at `TERMINAL_GROWTH` forever (Gordon growth model).
4. **Enterprise value**: all flows discounted at `DISCOUNT_RATE`.
5. **Equity value** = enterprise value − net debt, and **intrinsic value per share** = equity value / shares outstanding.
6. **Upside** = intrinsic value / price − 1.

All rates are **nominal, in reais**, since the cash flows include inflation. The discount rate should be close to the cost of capital of a Brazilian company, which is well above the Selic rate for most of them.

Leases are counted as debt, and lease payments are financing flows (IFRS 16), so they are not deducted from the FCF: the two choices are consistent.

In [7]:
pd.Series({
    "FCF_BASE_YEARS": FCF_BASE_YEARS,
    "PROJECTION_YEARS": PROJECTION_YEARS,
    "GROWTH_RATE": GROWTH_RATE,
    "TERMINAL_GROWTH": TERMINAL_GROWTH,
    "DISCOUNT_RATE": DISCOUNT_RATE,
}, name="assumption")

FCF_BASE_YEARS     3.00
PROJECTION_YEARS   5.00
GROWTH_RATE        0.05
TERMINAL_GROWTH    0.04
DISCOUNT_RATE      0.15
Name: assumption, dtype: float64

In [8]:
valuation = build_valuation(history_full, market)
print(f"Companies valued: {valuation['INTRINSIC_VALUE'].notna().sum()} of {len(valuation)}")

Companies valued: 129 of 193


## 6. Vivara

The first company of the project, used to validate every step. Its FCF and net debt were checked against the filings in `01_fcf_history.ipynb`.

In [9]:
vivara = valuation[valuation["CD_CVM"] == 24805]
vivara.T

,93
CD_CVM,24805
DENOM_CIA,VIVARA PARTICIPAÇÕES S.A.
TICKER,VIVA3
BASE_FCF,"144,458.59"
NET_DEBT,"829,822.81"
SHARES,"235,071,814.00"
INTRINSIC_VALUE,2.51
PRICE,27.03
UPSIDE,-0.91
AVG_TRADED_VALUE,"75,856,244.07"


### 6.1 Sensitivity

The intrinsic value changes a lot with the discount rate and the growth rate. The table shows the value per share for a range of both (the other assumptions keep their defaults).

In [10]:
row = vivara.iloc[0]
sensitivity_table(
    row["BASE_FCF"], row["NET_DEBT"], row["SHARES"],
    discount_rates=[0.12, 0.14, 0.15, 0.16, 0.18],
    growth_rates=[0.0, 0.05, 0.10, 0.15],
)

,g=0%,g=5%,g=10%,g=15%
r=12%,3.22,4.80,6.68,8.92
r=14%,1.90,3.12,4.58,6.30
r=15%,1.42,2.51,3.82,5.35
r=16%,1.02,2.01,3.18,4.56
r=18%,0.39,1.21,2.18,3.33


### 6.2 Reading the result

- Check how the base FCF compares with each year of the history: a base built on low-FCF years (for example, years of heavy investment in inventory or stores) gives a low value even for a profitable company.
- Compare the price with the sensitivity table: which growth rate would the market price require? That implied growth is often more informative than the intrinsic value itself.

In [11]:
history_full.loc[history_full["CD_CVM"] == 24805, ["YEAR", "CFO", "CAPEX", "FCF", "NET_DEBT", "SHARES"]]

,YEAR,CFO,CAPEX,FCF,NET_DEBT,SHARES
3393,2019,"65,509.82","-38,833.63","26,676.19","98,764.75",NaN
3762,2020,"252,093.02","-44,711.52","207,381.50","138,871.56","236,197,769.00"
4183,2021,"219,771.46","-104,077.87","115,693.59","211,127.06","236,917,769.00"
4621,2022,"63,309.07","-137,731.15","-74,422.08","404,390.08","235,590,157.00"
5069,2023,"201,468.84","-178,185.72","23,283.12","487,319.92","235,200,814.00"
5517,2024,"150,681.23","-129,019.62","21,661.61","691,011.17","235,063,179.00"
5956,2025,"468,874.31","-80,443.29","388,431.02","829,822.81","235,071,814.00"


## 7. Ranking

All investable companies with a valid valuation, from the most undervalued. Companies traded less than R$ 1 million per day are left out: their prices are less reliable and they are hard to buy.

In [12]:
is_liquid = valuation["AVG_TRADED_VALUE"] >= 1_000_000
ranking = valuation[is_liquid & valuation["INTRINSIC_VALUE"].notna()]
print(f"{len(ranking)} liquid companies with a valuation")
ranking.head(20)

94 liquid companies with a valuation


,CD_CVM,DENOM_CIA,TICKER,BASE_FCF,NET_DEBT,SHARES,INTRINSIC_VALUE,PRICE,UPSIDE,AVG_TRADED_VALUE
0,20362,POSITIVO TECNOLOGIA S.A.,POSI3,"347,948.67","678,290.00","139,211.00","19,713.59",5.08,"3,879.63","3,745,108.46"
1,11932,IOCHPE MAXION S.A.,MYPK3,"970,213.33","4,031,813.00","149,770.00","36,801.81",11.80,"3,117.80","8,604,263.62"
9,25038,LOJAS QUERO QUERO S.A.,LJQQ3,"168,001.33","493,124.00","206,917.00","5,603.41",3.05,"1,836.18","6,187,199.53"
10,20346,PROFARMA DISTRIB PROD FARMACEUTICOS S.A.,PFRM3,"276,480.67","755,485.00","122,611.00","16,019.36",9.59,"1,669.42","1,253,258.58"
11,19615,GRENDENE S.A.,GRND3,"516,845.33","-987,507.00","902,160.00","6,729.98",4.67,"1,440.11","12,513,657.61"
13,8133,LOJAS RENNER S.A.,LREN3,"1,860,090.33","982,555.00","982,199.00","17,628.25",14.58,"1,208.07","267,641,732.16"
15,6343,TUPY S.A.,TUPY3,"528,287.67","2,241,560.00","131,077.00","22,544.04",20.41,"1,103.56","13,274,577.45"
19,7870,KEPLER WEBER S.A.,KEPL3,"136,887.33","16,731.00","173,332.00","7,671.85",7.86,975.06,"9,666,731.96"
20,23264,AMBEV S.A.,ABEV3,"19,972,357.67","-16,932,996.00","15,616,526.00","13,664.59",16.68,818.22,"486,137,169.51"
21,21148,CONSTRUTORA TENDA S.A.,TEND3,"298,494.33","306,549.00","121,182.00","21,699.81",28.36,764.16,"114,787,338.45"


## 8. Limitations of the valuation

- **One set of assumptions for every company.** Growth and risk differ a lot across sectors; a utility and a retailer should not share the same discount rate. The ranking is a screen to find companies worth studying, not a final answer.
- **Historical FCF as the base.** Companies that are investing heavily today (low FCF now, higher later) look overvalued by the model, and cyclical companies at a peak of their cycle (high FCF that will not last) look undervalued.
- **FCF definition.** `CFO + CAPEX` from the cash flow statement. Interest paid may be inside the CFO for some companies, which lowers their FCF to the firm (see `01_fcf_history.ipynb`).
- **Same value for every share class.** Common and preferred shares are valued equally; in practice their prices may differ.
- **Units left out.** Companies are priced through their common or preferred shares, never through units.
- **Data errors.** The model inherits any error in the CVM filings (scale, signs, share count). Always check a company's numbers before relying on its result.